1. VERIFICAÇÃO DO BANCO DE DADOS

In [1]:
file.exists("/content/exportacao_cargas.csv.gz")

[1] FALSE

2. IMPORTAÇÃO DO BANCO DE DADOS

In [2]:
dados <- read.csv(
  "/content/exportacao_cargas.csv.gz",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

Warning message in file(file, "rt"):
“cannot open file '/content/exportacao_cargas.csv.gz': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


3. VERIFICANDO SE O BANCO DE DADOS FOI IMPORTADO (LINHAS E COLUNAS)

In [ ]:
dim(dados)

4. TRANSFORMANDO TOTAL_TONELADAS EM NÚMERO (TIRANDO A VÍRGUAL E COLOCANDO PONTO, (QUALITATIVA CONTÍNUA))

In [ ]:
dados$TOTAL_TONELADAS <- as.numeric(
  gsub(",", ".", dados$TOTAL_TONELADAS)
)

5. CORREÇÃO DE VARIÁVEIS NUMÉRICAS (QUANTITATIVA DISCRETA)

In [ ]:
dados$TOTAL_TEU <- as.integer(dados$TOTAL_TEU)
dados$TOTAL_UNID <- as.integer(dados$TOTAL_UNID)

6. CRIANDO SUBCONJUNTO PARA ANALISAR TOTAL_TEU PARA APENAS CARGA CONTERIZADA

In [ ]:
dados_container <- dados[
  dados$NATUREZA_CARGA == "CARGA CONTEINERIZADA",
]

table(dados_container$NATUREZA_CARGA)
cat("\n")

dim(dados_container)

7. CRIANDO VARIÁVEL Y COM CARGA CONTERIZA (NATUREZA_CARGA)

In [ ]:
dados$y <- as.integer(dados$NATUREZA_CARGA == "CARGA CONTEINERIZADA")
table(dados$y)

8. FAZENDO REGRESSÃO LINEAR SIMPLES

In [ ]:
m <- lm(TOTAL_TONELADAS ~ TOTAL_TEU, data = dados_container)
summary(m)

9. GRÁFICO QUADRADO COM RETA AJUSTADA

In [ ]:
par(pty = "s")

plot(
  dados_container$TOTAL_TEU, dados_container$TOTAL_TONELADAS,
  pch = 19, col = "blue",
  main = "TEU x Toneladas, Carga Conteinerizada",
  xlab = "Total de TEUs", ylab = "Total de toneladas"
)

abline(m, col = "orange", lwd = 3)

10. FAZENDO REGRESSÃO LINEAR MÚLTIPLA

In [ ]:
m2 <- lm(TOTAL_TONELADAS ~ TOTAL_TEU + TOTAL_UNID, data = dados_container)
summary(m2)

11. DIAGNÓSTICO DOS RESÍDUOS

In [ ]:
par(pty = "s")

plot(fitted(m2), resid(m2), pch = 19, col = "blue")
abline(h = 0, col = "orange", lwd = 3, lty = 2)

12. APLICANDO LOG

In [ ]:
dados_container$log_TEU <- log(dados_container$TOTAL_TEU)
dados_container$log_TONELADAS <- log(dados_container$TOTAL_TONELADAS)

13. REGRESSÃO COM VARIÁVEIS EM LOG

In [ ]:
m_log <- lm(log_TONELADAS ~ log_TEU, data = dados_container)
summary(m_log)  # b0, b1, p-valor, R²

14. GRÁFICO QUADRADO COM RETA EM LOG

In [3]:
par(pty = "s")

plot(
  dados_container$log_TEU, dados_container$log_TONELADAS,
  pch = 19, col = "blue",
  main = "log(TEU) x log(Toneladas), Carga Conteinerizada",
  xlab = "log(Total de TEUs)", ylab = "log(Total de toneladas)"
)

abline(m_log, col = "orange", lwd = 3)

ERROR: Error: object 'dados_container' not found


15. DIAGNÓSTICO COM RESÍDUOS EM LOG

In [ ]:
par(pty = "s")

plot(fitted(m_log), resid(m_log), pch = 19, col = "blue")
abline(h = 0, col = "orange", lwd = 3, lty = 2)

16. COMPARANDO O R² DOS DOIS MODELOS

In [ ]:
cat("R² modelo original:", summary(m)$r.squared, "\n")
cat("R² modelo em log:", summary(m_log)$r.squared, "\n")

17. INSTALANDO E CARREGANDO PACOTE IMTEST

In [ ]:
install.packages("lmtest")
library(lmtest)

18. TESTANDO BREUSCH-PAGAN NO MODELO ORIGINAL

In [ ]:
bptest(m)

19. TESTANDO BREUSCH-PAGAN NO MODELO EM LOG

In [ ]:
bptest(m_log)

20. CRIANDO X1 (TOTAL_TONELADAS) E X2 (TOTAL_UNID)

In [ ]:
dados$x1 <- dados$TOTAL_TONELADAS
dados$x2 <- dados$TOTAL_UNID

21. REDUZINDO PARA UMA AMOSTRA

In [ ]:
set.seed(123)
dados <- dados[sample(nrow(dados), 5000), ]

22. AJUSTANDO A LOGÍSTICA E LENDO RAZÕES DE CHANCE

In [ ]:
m <- glm(y ~ x1 + x2, family = binomial, data = dados)
summary(m);  exp(coef(m))

23. PREVENDO PROBABILIDADE E CLASSE (LIMIAR 0.5)

In [4]:
p    <- predict(m, type = "response")
yhat <- as.integer(p > 0.5)
table(real = dados$y, previsto = yhat)

ERROR: Error: object 'm' not found


24. REPETINDO COM LIMIAR 0.3 E 0.7

In [ ]:
yhat03 <- as.integer(p > 0.3)
table(real = dados$y, previsto = yhat03)

yhat07 <- as.integer(p > 0.7)
table(real = dados$y, previsto = yhat07)

25. AUC

In [ ]:
mean(outer(p[dados$y == 1], p[dados$y == 0], ">"))

26. COMEÇANDO A DIVISÃO 70/30

In [ ]:
dados_reg <- subset(
  dados,
  NATUREZA_CARGA == "CARGA CONTEINERIZADA"
)

27. CONFERINDO AS VARIÁVEIS

In [ ]:
summary(
  dados_reg[, c("TOTAL_TONELADAS",
                "TOTAL_TEU",
                "TOTAL_UNID")]
)

28. RETIRANDO POSSÍVEIS VALORES AUSENTES

In [ ]:
dados_reg <- na.omit(
  dados_reg[, c("TOTAL_TONELADAS",
                "TOTAL_TEU",
                "TOTAL_UNID")]
)

29. DIVIDINDO OS DADOS EM 70% (TREINAMENTO) E 30% (TESTE)

In [ ]:
set.seed(1)

itr <- sample(
  nrow(dados_reg),
  round(0.7 * nrow(dados_reg))
)

tr <- dados_reg[itr, ]
te <- dados_reg[-itr, ]

30. CONFERINDO A DIVISÃO

In [ ]:
nrow(tr)
nrow(te)
cat("\n")

nrow(tr) / nrow(dados_reg)
nrow(te) / nrow(dados_reg)

31. MODELO 1 (REGRESSAO SIMPLES)

In [5]:
m1 <- lm(
  TOTAL_TONELADAS ~ TOTAL_TEU,
  data = tr
)

summary(m1)

ERROR: Error in eval(mf, parent.frame()): object 'tr' not found


32. MODELO 2 (REGRESSÃO MÚLTIPLA)

In [ ]:
m2 <- lm(
  TOTAL_TONELADAS ~ TOTAL_TEU + TOTAL_UNID,
  data = tr
)

summary(m2)

33. COMPARANDO R² DOS MODELOS

In [ ]:
cat("R² Modelo 1 (simples): ", summary(m1)$r.squared, "\n")
cat("R² Modelo 2 (múltiplo): ", summary(m2)$r.squared, "\n")

34. FAZENDO PREVISÕES NO BANCO DE TESTE

In [ ]:
pred1 <- predict(m1, newdata = te)

pred2 <- predict(m2, newdata = te)

35. CALCULANDO O RMSE (RAIZ DO ERRO QUADRÁTICO MÉDIO)

In [ ]:
mse1 <- mean(
  (te$TOTAL_TONELADAS - pred1)^2
)

mse2 <- mean(
  (te$TOTAL_TONELADAS - pred2)^2
)

rmse1 <- sqrt(mse1)
rmse2 <- sqrt(mse2)

cat("RMSE Modelo 1 (simples): ", rmse1, "\n")
cat("RMSE Modelo 2 (múltiplo): ", rmse2, "\n")